# Когорты и отток

В `01_data_quality.ipynb` мы выяснили, какие когорты вообще можно считать: только с июля 2015
по апрель 2016 и только тех клиентов, чей первый месяц в панели совпал с месяцем подключения.
Это 127 034 клиента.

Определения, с которыми работаем:

- **когорта** - месяц `fecha_alta`, то есть месяц, когда клиент стал держателем первого договора;
- **активный клиент** - в этом месяце у него есть хотя бы один продукт. Просто присутствие в
  выгрузке ничего не значит: четверть строк панели - клиенты без единого продукта;
- **отток продукта** - флаг был 1 в месяце N и 0 в N+1;
- **мигание** - тот же уход, но в N+2 флаг снова 1. Это не отток, а перерыв, и считать его надо отдельно.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.functions import (COHORT_FIRST, COHORT_LAST, PRODUCT_NAMES,
                           build_panel, cohort_members, product_transitions,
                           retention_matrix)

pd.set_option('display.width', 250)
sns.set_theme(style='whitegrid')
FIGURES = PROJECT_ROOT / 'reports' / 'figures'

con = duckdb.connect(config={'threads': 6})
build_panel(con, PROJECT_ROOT / 'data' / 'processed' / 'train.parquet')
cohort_members(con, COHORT_FIRST, COHORT_LAST)
con.execute('SELECT count(*) AS членов_когорт FROM mem').fetchdf()

## Первый месяц жизни считать нельзя

Прежде чем строить матрицу, проверим неприятную вещь. Клиент подключился в этом месяце - значит
ли это, что в снимке за тот же месяц у него уже есть продукт?

In [ ]:
con.execute("""
WITH k0 AS (SELECT mem.ncodpers, mem.cohort, p.n_prod AS p0
            FROM mem JOIN p ON p.ncodpers = mem.ncodpers AND p.m = mem.cohort),
     k1 AS (SELECT mem.ncodpers, p.n_prod AS p1
            FROM mem JOIN p ON p.ncodpers = mem.ncodpers AND p.m = mem.cohort + INTERVAL 1 MONTH)
SELECT count(*) AS всего,
       count(*) FILTER (WHERE p0 = 0) AS ноль_на_k0,
       count(*) FILTER (WHERE p0 = 0 AND p1 >= 1) AS продукт_появился_на_k1,
       count(*) FILTER (WHERE p0 = 0 AND coalesce(p1, 0) = 0) AS остались_нулевыми
FROM k0 LEFT JOIN k1 USING (ncodpers)""").fetchdf()

У 51 379 клиентов из 127 034 в месяц подключения нет ни одного продукта. У 14 900 из них продукт
появляется на следующий месяц - то есть договор оформлен, а в снимок он попал позже. А 36 479 так
и остаются без продуктов.

Значит первый месяц жизни (k=0) как база для удержания не годится: он занижен из-за задержки
регистрации. Будем считать удержание от второго месяца (k=1), а k=0 показывать отдельно.

## Когортная матрица удержания

In [ ]:
от_когорты = retention_matrix(con, base='cohort')
от_второго = retention_matrix(con, base='k1')
print('Активные клиенты, % от размера когорты:')
display(от_когорты)
print('\nАктивные клиенты, % от активных на втором месяце:')
display(от_второго)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(от_второго.drop(columns=[0]), annot=True, fmt='.1f', cmap='YlGnBu',
            vmin=90, vmax=101, cbar_kws={'label': '% от активных на k=1'}, ax=ax)
ax.set_title('Удержание активных клиентов по когортам подключения')
ax.set_xlabel('месяц жизни (k)')
ax.set_ylabel('когорта подключения')
fig.tight_layout()
fig.savefig(FIGURES / 'cohort_retention.png', dpi=150)
plt.show()

Матрица почти плоская: от второго месяца жизни к десятому когорты теряют 4-5% активных клиентов,
а когорта октября 2015 - меньше процента. Через 10 месяцев у июльской когорты активны 96,1%.

Это важный и неожиданный результат: **на уровне клиента оттока почти нет**. Причина видна в
первой матрице - удержание считается от тех, кто держит хотя бы один продукт, а этот продукт
почти всегда текущий счёт, который никто не закрывает.

Значит вопрос «кто уходит» надо задавать не про клиента, а про продукт.

## Отток по продуктам

Считаем переходы 1 → 0 по всей панели и сразу отделяем мигания (1 → 0 → 1).

In [ ]:
tr = product_transitions(con)
tr.head(12)

In [ ]:
top = tr.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(top['продукт'], top['чистый_отток'], label='ушли и не вернулись')
ax.barh(top['продукт'], top['мигание'], left=top['чистый_отток'], label='мигание 1 → 0 → 1')
ax.set_xlabel('переходов 1 → 0 за 17 месяцев')
ax.set_title('Уходы из продукта: сколько из них реальные')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / 'product_churn.png', dpi=150)
plt.show()

Вот где живёт отток. Больше всего уходов у автоплатежа (136 594), текущего счёта (89 644) и
зачисления пенсии (76 112).

Но доля миганий у этих продуктов совсем разная:

- зачисление пенсии - 55,8% «уходов» возвращаются в следующем месяце;
- зачисление зарплаты - 50,0%;
- автоплатёж - 37,7%;
- а вот у депозитов, ипотеки и пенсионного плана - 0,2-1,1%.

Логика понятная: зарплату и пенсию могли просто не зачислить в конкретном месяце, и флаг погас,
хотя клиент никуда не ушёл. А закрытый депозит закрыт окончательно.

Практический вывод, который стоит того, чтобы его проговорить: **единого определения оттока для
всех продуктов быть не может**. По зарплатно-пенсионным продуктам отток надо считать с окном
(нет флага 2-3 месяца подряд), иначе цифра завышается примерно вдвое. По депозитам и кредитам
достаточно одного месяца.

## Когда именно уходят

Теперь посмотрим на отток по месяцу жизни когорты: считаем только тех, у кого в прошлом месяце
был хотя бы один продукт.

In [ ]:
con.execute("""CREATE OR REPLACE TABLE w AS
SELECT ncodpers, m, n_prod, antiguedad,
       lag(n_prod) OVER (PARTITION BY ncodpers ORDER BY m) AS prev
FROM p""")

by_k = con.execute("""
SELECT datediff('month', mem.cohort, w.m) AS k,
       count(*) FILTER (WHERE w.prev >= 1) AS держателей,
       round(100.0*count(*) FILTER (WHERE w.prev >= 1 AND w.n_prod < w.prev)
             /nullif(count(*) FILTER (WHERE w.prev >= 1), 0), 2) AS потеряли_продукт_проц,
       round(100.0*count(*) FILTER (WHERE w.prev >= 1 AND w.n_prod = 0)
             /nullif(count(*) FILTER (WHERE w.prev >= 1), 0), 2) AS обнулились_проц
FROM mem JOIN w USING (ncodpers) WHERE w.m > mem.cohort
GROUP BY 1 ORDER BY 1""").fetchdf()
by_k

Отток продукта нарастает до четвёртого месяца жизни (2,11% → 3,22%), а потом спадает до 1,5%
к десятому. То есть опасное окно - это не первый месяц, а третий-четвёртый: клиент попробовал
продукт, не втянулся и отказался.

Полное обнуление набора, наоборот, максимально сразу (1,73% на k=1) и дальше падает в три раза.

Для сравнения - отток у «старых» клиентов, которые подключились до начала окна. Их 794 551,
и статистика там в разы богаче.

In [ ]:
con.execute("""
WITH old AS (SELECT ncodpers FROM fs WHERE cohort < DATE '2015-01-01')
SELECT CASE WHEN w.antiguedad < 12 THEN '1. меньше года'
            WHEN w.antiguedad < 36 THEN '2. 1-3 года'
            WHEN w.antiguedad < 72 THEN '3. 3-6 лет'
            WHEN w.antiguedad < 120 THEN '4. 6-10 лет'
            WHEN w.antiguedad IS NULL THEN '6. стаж неизвестен'
            ELSE '5. 10+ лет' END AS стаж,
       count(*) FILTER (WHERE w.prev >= 1) AS месяцев_с_продуктом,
       round(100.0*count(*) FILTER (WHERE w.prev >= 1 AND w.n_prod < w.prev)
             /nullif(count(*) FILTER (WHERE w.prev >= 1), 0), 2) AS теряют_продукт_проц,
       round(100.0*count(*) FILTER (WHERE w.prev >= 1 AND w.n_prod = 0)
             /nullif(count(*) FILTER (WHERE w.prev >= 1), 0), 2) AS обнуляются_проц
FROM w JOIN old USING (ncodpers) WHERE w.prev IS NOT NULL
GROUP BY 1 ORDER BY 1""").fetchdf()

Любопытно: чем дольше клиент с банком, тем **чаще** он теряет отдельный продукт (4,73% в месяц
у стажа 10+ лет против 2,64% у 3-6 лет), но тем **реже** обнуляет набор целиком (0,22% против
0,27%). Это не противоречие - у старых клиентов просто больше продуктов, поэтому вероятность
потерять хоть один выше, а уйти совсем - ниже.

Строка «стаж неизвестен» с оттоком 28,66% - это тот самый блок битых строк из первого ноутбука,
628 наблюдений. В выводы не берём.

## Гипотеза 1: cross-sell защищает от ухода

Берём клиентов, активных на втором месяце жизни, и смотрим, кто обнулит набор за следующие
полгода.

In [ ]:
con.execute("""CREATE OR REPLACE TABLE base AS
SELECT mem.ncodpers, mem.cohort, p.n_prod AS p1,
       coalesce(TRY_CAST(trim(CAST(t.ind_recibo_ult1 AS VARCHAR)) AS INT), 0) AS автоплатёж,
       p.segmento, p.canal
FROM mem
JOIN p ON p.ncodpers = mem.ncodpers AND p.m = mem.cohort + INTERVAL 1 MONTH
JOIN t ON t.ncodpers = mem.ncodpers
      AND date_trunc('month', t.fecha_dato) = mem.cohort + INTERVAL 1 MONTH
WHERE p.n_prod >= 1 AND mem.cohort <= DATE '2015-10-01'""")

con.execute("""CREATE OR REPLACE TABLE fut AS
SELECT base.ncodpers, max(CASE WHEN p.n_prod = 0 THEN 1 ELSE 0 END) AS обнулился
FROM base JOIN p ON p.ncodpers = base.ncodpers
     AND p.m > base.cohort + INTERVAL 1 MONTH
     AND p.m <= base.cohort + INTERVAL 7 MONTH
GROUP BY 1""")

con.execute("""
SELECT CASE WHEN b.p1 = 1 THEN '1 продукт' WHEN b.p1 = 2 THEN '2 продукта'
            ELSE '3 и больше' END AS набор,
       count(*) AS клиентов, round(100.0*avg(f.обнулился), 1) AS обнулились_проц
FROM base b JOIN fut f USING (ncodpers) GROUP BY 1 ORDER BY 1""").fetchdf()

Результат странный: 1 продукт - 5,0%, 2 продукта - 9,7%, 3 и больше - 2,1%. Немонотонно, и
гипотеза как будто не работает. Но прежде чем это записывать, проверим состав групп - вдруг
они отличаются не числом продуктов, а тем, что это за клиенты.

In [ ]:
con.execute("""
SELECT coalesce(b.segmento, '(пусто)') AS сегмент,
       CASE WHEN b.p1 = 1 THEN '1 продукт' WHEN b.p1 = 2 THEN '2 продукта'
            ELSE '3 и больше' END AS набор,
       count(*) AS клиентов, round(100.0*avg(f.обнулился), 1) AS обнулились_проц
FROM base b JOIN fut f USING (ncodpers)
GROUP BY 1, 2 ORDER BY 1, 2""").fetchdf()

Вот в чём было дело. Внутри каждого сегмента картина другая:

| сегмент | 1 продукт | 2 продукта | 3 и больше |
|---|---|---|---|
| 01 - TOP (VIP) | 34,8% | 25,8% | 9,5% |
| 02 - PARTICULARES (физлица) | 8,5% | 8,4% | 1,8% |
| 03 - UNIVERSITARIO (студенты) | 3,8% | 4,5% | 1,4% |

Три и больше продуктов защищают в каждом сегменте - от 1,4% до 9,5% против 3,8-34,8% у
одиночного продукта. А агрегат переворачивался потому, что группа «1 продукт» на 77% состоит из
студентов (44 182 из 57 358), у которых отток низкий сам по себе, а «2 продукта» - в основном
физлица (2256 из 3108), которые уходят чаще.

Это парадокс Симпсона в чистом виде: не учли сегмент - получили обратный вывод.

Второй разрез - что именно за пара продуктов.

In [ ]:
con.execute("""
SELECT CASE WHEN b.автоплатёж = 1 THEN 'текущий счёт + автоплатёж' ELSE 'другая пара' END AS набор,
       count(*) AS клиентов, round(100.0*avg(f.обнулился), 1) AS обнулились_проц
FROM base b JOIN fut f USING (ncodpers) WHERE b.p1 = 2
GROUP BY 1 ORDER BY 2 DESC""").fetchdf()

И внутри «двух продуктов» состав пары решает почти всё: текущий счёт плюс автоплатёж - 3,3%
обнулившихся, любая другая пара - 16,6%, разница в пять раз. Автоплатёж привязывает клиента:
по нему проходят регулярные списания, и закрыть счёт становится неудобно.

**Гипотеза подтверждается, но с оговоркой:** дело не в числе продуктов само по себе, а в том,
какие это продукты и кому они проданы. Сравнивать группы по числу продуктов без контроля
сегмента нельзя.

## Гипотеза 2: канал привлечения определяет удержание

In [ ]:
con.execute("""
WITH m2 AS (SELECT * FROM mem WHERE cohort <= DATE '2015-10-01'),
     ch AS (SELECT m2.ncodpers, arg_min(p.canal, p.m) AS канал,
                   arg_min(p.segmento, p.m) AS сегмент
            FROM m2 JOIN p USING (ncodpers) GROUP BY 1),
     k6 AS (SELECT m2.ncodpers, max(CASE WHEN p.n_prod >= 1 THEN 1 ELSE 0 END) AS активен
            FROM m2 JOIN p ON p.ncodpers = m2.ncodpers AND p.m = m2.cohort + INTERVAL 6 MONTH
            GROUP BY 1)
SELECT coalesce(ch.канал, '(пусто)') AS канал, count(*) AS клиентов,
       round(100.0*avg(k6.активен), 1) AS активны_через_6_мес_проц,
       mode(ch.сегмент) AS преобладающий_сегмент
FROM ch JOIN k6 USING (ncodpers)
GROUP BY 1 HAVING count(*) >= 300
ORDER BY 2 DESC LIMIT 8""").fetchdf()

Разброс между каналами есть, и он заметный: KHO даёт 90,1% активных через полгода, KHN - 87,0%,
KHM - 79,8%, а самый массовый KHQ (63 030 клиентов из 78 тыс.) - только 76,1%.

Но осторожно: канал и сегмент в этих данных связаны, KHQ - это в основном студенты. Поэтому
честная формулировка такая: канал KHQ приводит много клиентов с одним продуктом и низкой
активностью, и вопрос «это канал плохой или аудитория такая» на одних этих данных не решается.
Для ответа нужен разрез канал × сегмент с достаточным числом наблюдений в каждой клетке, а у нас
почти все студенты пришли через один канал.

## Гипотеза 3: сегмент важнее всего

In [ ]:
con.execute("""
WITH m2 AS (SELECT * FROM mem WHERE cohort <= DATE '2015-10-01'),
     seg AS (SELECT m2.ncodpers, arg_min(p.segmento, p.m) AS сегмент
             FROM m2 JOIN p USING (ncodpers) GROUP BY 1),
     k6 AS (SELECT m2.ncodpers, max(CASE WHEN p.n_prod >= 1 THEN 1 ELSE 0 END) AS активен
            FROM m2 JOIN p ON p.ncodpers = m2.ncodpers AND p.m = m2.cohort + INTERVAL 6 MONTH
            GROUP BY 1)
SELECT coalesce(seg.сегмент, '(пусто)') AS сегмент, count(*) AS клиентов,
       round(100.0*avg(k6.активен), 1) AS активны_через_6_мес_проц
FROM seg JOIN k6 USING (ncodpers) GROUP BY 1 ORDER BY 2 DESC""").fetchdf()

Через полгода активны 84,7% физлиц, 82,6% VIP-клиентов и 74,9% студентов. Студенты - самая
массовая группа новых когорт (59 243 из 78 тыс.) и самая слабая по активности.

При этом у студентов, как мы видели выше, самый низкий риск обнулиться (3,8% против 8,5% у
физлиц). Разница в том, что «активен» здесь - это наличие хотя бы одного продукта, а обнуление -
потеря всех. Студент чаще приходит с пустым набором и так в нём и остаётся, а физлицо приходит
с продуктом и может его потерять.

## Выводы

1. **На уровне клиента оттока почти нет.** Когорты июля 2015 - апреля 2016 удерживают 96-100%
   активных клиентов от второго месяца к десятому. Причина - текущий счёт, который есть почти у
   всех и который никто не закрывает.
2. **Отток живёт на уровне продуктов.** Лидеры по уходам: автоплатёж (136 594 перехода 1 → 0),
   текущий счёт (89 644), зачисление пенсии (76 112).
3. **Единого определения оттока для всех продуктов нет.** У зачисления пенсии 55,8% «уходов» -
   мигание флага, у зарплаты 50,0%, у автоплатежа 37,7%, а у депозитов и ипотеки - меньше 1%.
   По зарплатно-пенсионным продуктам отток надо считать с окном в 2-3 месяца, иначе метрика
   завышается вдвое.
4. **Опасное окно - третий и четвёртый месяц жизни**, а не первый: отток продукта растёт с 2,11%
   до 3,22% и потом спадает до 1,5%.
5. **Cross-sell защищает, но считать его надо внутри сегмента.** Три продукта и больше снижают
   риск обнуления в каждом сегменте, а наивный агрегат даёт обратный ответ из-за парадокса
   Симпсона. Состав набора важнее количества: пара «текущий счёт + автоплатёж» - 3,3% обнулений
   против 16,6% у любой другой пары.
6. **Что с этим делать банку:** продавать второй продукт не любой, а привязывающий (автоплатёж,
   зачисление зарплаты); работать с клиентом на третий-четвёртый месяц, а не в момент
   подключения; не считать погасший флаг зарплаты оттоком, пока не прошло 2-3 месяца.

### Оговорки

- Панель короткая: 17 месяцев, и у поздних когорт видно только несколько месяцев жизни.
- Когорты до июля 2015 исключены: до этого месяца выгрузка отдавала не всех клиентов.
- Май 2016 в расчёте оттока не участвует - для последнего месяца отток не определён.
- Канал и сегмент связаны, разделить их влияние на этих данных нельзя.
- 794 551 клиент подключился до начала окна, их первые месяцы жизни в данные не попали; мы
  использовали их только для разреза по стажу.